<h1>Trabalho 2 — Análise Probabilística NBA</h1>

**Disciplina:** Métodos Quantitativos para Computação — UNIFOR  
**Professor:** Prof. Me. Ricardo Carubbi  
**Equipe:** Milwaukee Bucks (MIL)  

---

## Pergunta de Pesquisa

> **A não permanência observada no mesmo time é mais frequente entre casos jogador-temporada classificados como baixo desempenho?**

---

## Eventos definidos

- **S**: caso jogador-temporada em que o jogador **não é observado no mesmo time na temporada seguinte** (`nao_permanencia = True`).
- **B**: caso jogador-temporada com **baixo desempenho**, definido por `X ≥ 2` indicadores de baixo desempenho simultâneos.

---

---
# PESSOA 1 — Preparação dos Dados + Associação Quantitativa
---

## Bloco 1 — Importação de bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams.update({
    'figure.figsize': (8, 5),
    'font.size': 11,
    'axes.titlesize': 13,
})

## Bloco 2 — Carregamento do dataset

In [ ]:
df = pd.read_csv('dataset/nba_stats_preprocessed.csv')

print(f'Shape: {df.shape}')
print(f'Temporadas disponíveis: {sorted(df["temporada"].unique())}')
df.head()

## Bloco 3 — Criação do identificador `jogador_temporada`

In [ ]:
# Identificador auxiliar: combina id_jogador e temporada
df['jogador_temporada'] = df['id_jogador'].astype(str) + '_' + df['temporada'].astype(str)

print('Exemplo de jogador_temporada:')
df[['id_jogador', 'temporada', 'jogador_temporada']].head(5)

## Bloco 4 — Construção da variável `nao_permanencia`

Para cada caso jogador-temporada, verificamos se o mesmo `id_jogador` aparece no mesmo `sigla_time` na temporada seguinte.

- `nao_permanencia = False` → jogador observado no mesmo time na temporada seguinte.
- `nao_permanencia = True`  → jogador não observado no mesmo time na temporada seguinte.
- Casos da **última temporada disponível** são excluídos da análise, pois não há temporada seguinte observável.

In [ ]:
ultima_temporada = df['temporada'].max()
print(f'Última temporada disponível: {ultima_temporada}')

# Cria tabela de referência: id_jogador + sigla_time para cada temporada T+1
proxima_temp = (
    df[['id_jogador', 'sigla_time', 'temporada']]
    .copy()
    .rename(columns={'temporada': 'temporada_seguinte', 'sigla_time': 'time_seguinte'})
)

# Faz o merge: para cada caso jogador-temporada T, busca se aparece no mesmo time em T+1
df_merge = df.merge(
    proxima_temp,
    left_on=['id_jogador', 'temporada'],
    right_on=['id_jogador', 'temporada_seguinte'],
    how='left'
)

# nao_permanencia: True se não aparece no mesmo time na temporada seguinte
df_merge['nao_permanencia'] = ~(
    (df_merge['time_seguinte'] == df_merge['sigla_time']) &
    (df_merge['temporada_seguinte'] == df_merge['temporada'] + 1)
)

# Remove casos da última temporada (sem temporada seguinte observável)
df_analise = df_merge[df_merge['temporada'] < ultima_temporada].copy()

print(f'Total de casos no dataset completo:  {len(df)}')
print(f'Casos removidos (última temporada):  {len(df[df["temporada"] == ultima_temporada])}')
print(f'Casos no subconjunto de análise:     {len(df_analise)}')
print(f'\nDistribuição de nao_permanencia:')
print(df_analise['nao_permanencia'].value_counts())
print(f'\nProporção de não permanência: {df_analise["nao_permanencia"].mean():.3f}')

## Bloco 5 — Parte 1: Associação Quantitativa

Analisamos a associação linear entre `pontos`, `minutos_totais`, `jogos_disputados` e `salario_usd` usando covariância, correlação de Pearson e visualizações.

### 5.1 — Heatmap de correlação

In [ ]:
cols_assoc = ['pontos', 'minutos_totais', 'jogos_disputados', 'salario_usd']

corr_matrix = df_analise[cols_assoc].corr(method='pearson')

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    square=True, linewidths=0.5,
    ax=ax
)
ax.set_title('Matriz de correlação de Pearson\n(dataset completo — casos jogador-temporada)')
plt.tight_layout()
plt.show()

print('\nMatriz de correlação:')
display(corr_matrix.round(3))

**Interpretação:** O heatmap revela que `pontos`, `minutos_totais` e `jogos_disputados` apresentam correlações positivas entre si — jogadores que disputam mais jogos tendem a acumular mais minutos e pontos. A correlação entre `pontos` e `salario_usd` é positiva, sugerindo que maior pontuação está associada a maiores salários. Contudo, a correlação indica apenas associação linear, e não deve ser interpretada como relação de causa e efeito.

### 5.2 — Gráfico de dispersão: `pontos` vs `salario_usd`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    df_analise['pontos'], df_analise['salario_usd'],
    alpha=0.25, color='steelblue', edgecolors='none', s=20
)

# Linha de tendência
z = np.polyfit(df_analise['pontos'].dropna(), df_analise['salario_usd'].dropna(), 1)
p = np.poly1d(z)
xl = np.linspace(df_analise['pontos'].min(), df_analise['pontos'].max(), 200)
ax.plot(xl, p(xl), color='red', linestyle='--', linewidth=1.5, label='Tendência linear')

ax.set_title('Dispersão: Pontos vs Salário (USD)\n(dataset completo — casos jogador-temporada)')
ax.set_xlabel('Pontos na temporada')
ax.set_ylabel('Salário (USD)')
ax.legend()
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

### 5.3 — Covariância e correlação de Pearson: `pontos` vs `salario_usd`

In [ ]:
par = df_analise[['pontos', 'salario_usd']].dropna()

# Covariância amostral
cov_manual = ((par['pontos'] - par['pontos'].mean()) * (par['salario_usd'] - par['salario_usd'].mean())).sum() / (len(par) - 1)
cov_pandas = par['pontos'].cov(par['salario_usd'])

# Correlação de Pearson
r, p_valor = stats.pearsonr(par['pontos'], par['salario_usd'])

# Coeficiente de determinação R²
r2 = r ** 2

print(f'Covariância amostral (manual):  {cov_manual:,.2f}')
print(f'Covariância amostral (pandas):  {cov_pandas:,.2f}')
print(f'Correlação de Pearson (r):      {r:.4f}')
print(f'Coeficiente de determinação R²: {r2:.4f}  ({r2*100:.2f}% da variância explicada)')
print(f'p-valor:                        {p_valor:.4e}')

**Interpretação:** A covariância positiva indica que pontuações acima da média tendem a ocorrer junto com salários acima da média. A correlação de Pearson (r) quantifica essa associação numa escala padronizada entre -1 e 1. O coeficiente de determinação R² = r² indica a proporção da variabilidade do salário que é linearmente associada à variabilidade dos pontos — por exemplo, R² = 0,25 significa que 25% da variância do salário está associada linearmente à pontuação. O restante da variância é atribuído a outros fatores não incluídos no modelo, como posição, experiência e mercado. Não é possível afirmar causalidade com base nessa análise.

## Bloco 5.4 — Paradoxo de Simpson: correlação pontos × salário estratificada por posição

O **Paradoxo de Simpson** ocorre quando uma tendência observada no conjunto agregado desaparece ou se inverte ao estratificar os dados por uma variável de confusão.

Aqui investigamos se a correlação positiva entre `pontos` e `salario_usd` observada no dataset completo se mantém dentro de cada posição (`sigla_posicao`). A posição é uma candidata natural à variável de confusão porque:
- Centros (C) tendem a ter salários elevados independentemente de pontuação;
- Guards (PG, SG) pontuam mais mas nem sempre recebem os maiores salários;
- Cada posição tem distribuição de pontos e salários própria, o que pode distorcer a correlação agregada.

In [ ]:
from scipy import stats

posicoes = sorted(df_analise['sigla_posicao'].dropna().unique())
cores_pos = {'PG': '#378ADD', 'SG': '#1D9E75', 'SF': '#D85A30', 'PF': '#D4537E', 'C': '#7F77DD'}

# Correlação global
par_global = df_analise[['pontos', 'salario_usd']].dropna()
r_global, _ = stats.pearsonr(par_global['pontos'], par_global['salario_usd'])

# Correlação por posição
resultados = []
for pos in posicoes:
    sub = df_analise[df_analise['sigla_posicao'] == pos][['pontos', 'salario_usd']].dropna()
    if len(sub) > 5:
        r_pos, p_pos = stats.pearsonr(sub['pontos'], sub['salario_usd'])
        resultados.append({'Posição': pos, 'n': len(sub), 'r (Pearson)': round(r_pos, 4), 'p-valor': round(p_pos, 4)})

df_simpson = pd.DataFrame(resultados)
print(f'Correlação global (dataset completo): r = {r_global:.4f}')
print()
print('Correlação por posição (estratificada):')
display(df_simpson)

In [ ]:
# Gráfico: dispersão pontos × salário colorida por posição + tendência por estrato
fig, ax = plt.subplots(figsize=(9, 6))

for pos in posicoes:
    sub = df_analise[df_analise['sigla_posicao'] == pos][['pontos', 'salario_usd']].dropna()
    if len(sub) < 5:
        continue
    cor = cores_pos.get(pos, 'gray')
    ax.scatter(sub['pontos'], sub['salario_usd'],
               alpha=0.3, s=18, color=cor, label=f'{pos} (n={len(sub)})')
    # Linha de tendência por posição
    z = np.polyfit(sub['pontos'], sub['salario_usd'], 1)
    xl = np.linspace(sub['pontos'].min(), sub['pontos'].max(), 100)
    ax.plot(xl, np.poly1d(z)(xl), color=cor, linewidth=2)

# Linha de tendência global
z_g = np.polyfit(par_global['pontos'], par_global['salario_usd'], 1)
xl_g = np.linspace(par_global['pontos'].min(), par_global['pontos'].max(), 200)
ax.plot(xl_g, np.poly1d(z_g)(xl_g), color='black', linewidth=2,
        linestyle='--', label=f'Global (r={r_global:.2f})')

ax.set_title('Paradoxo de Simpson — Pontos vs Salário\nTendência global vs tendência por posição')
ax.set_xlabel('Pontos na temporada')
ax.set_ylabel('Salário (USD)')
ax.legend(fontsize=9, loc='upper left')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

# Gráfico de barras: r global vs r por posição
fig2, ax2 = plt.subplots(figsize=(7, 4))
labels = ['Global'] + df_simpson['Posição'].tolist()
valores_r = [r_global] + df_simpson['r (Pearson)'].tolist()
cores_bar = ['black'] + [cores_pos.get(p, 'gray') for p in df_simpson['Posição']]

bars = ax2.bar(labels, valores_r, color=cores_bar, alpha=0.8, edgecolor='white', width=0.55)
ax2.axhline(0, color='gray', linewidth=0.8)
ax2.axhline(r_global, color='black', linestyle='--', linewidth=1, alpha=0.5)
for bar, val in zip(bars, valores_r):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + (0.01 if val >= 0 else -0.03),
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax2.set_title('Correlação de Pearson: global vs por posição\n(Paradoxo de Simpson)')
ax2.set_ylabel('r (Pearson)')
ax2.set_ylim(-0.3, 1.0)
ax2.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

**Interpretação — Paradoxo de Simpson:** A correlação agregada entre `pontos` e `salario_usd` pode sugerir uma associação linear positiva no dataset completo. Contudo, ao estratificar por posição (`sigla_posicao`), observamos que a intensidade — e possivelmente a direção — dessa correlação varia entre os estratos. Isso caracteriza o Paradoxo de Simpson: a tendência global é influenciada pela composição dos grupos (posições), e não reflete necessariamente o padrão dentro de cada um deles. A posição funciona como variável de confusão porque determina tanto o perfil de pontuação esperado quanto o patamar salarial do atleta, independentemente do desempenho individual. Portanto, análises agregadas podem ser enganosas quando a variável de confusão não é controlada.

---
# PESSOA 2 — Variável Aleatória Discreta + Probabilidade Condicional + Conclusão
---

## Bloco 6 — Parte 2: Variável Aleatória Discreta X

Construímos X = número de indicadores de baixo desempenho observados no caso jogador-temporada.

Os três indicadores usam os quartis calculados sobre o dataset completo (subconjunto de análise):
- `ind_pontos`: 1 se `pontos` está no quartil inferior (< Q1 de pontos)
- `ind_minutos`: 1 se `minutos_totais` está no quartil inferior (< Q1 de minutos_totais)
- `ind_jogos`: 1 se `jogos_disputados` está no quartil inferior (< Q1 de jogos_disputados)

$$X = \text{ind\_pontos} + \text{ind\_minutos} + \text{ind\_jogos} \in \{0, 1, 2, 3\}$$

### 6.1 — Construção dos indicadores e de X

In [ ]:
# Quartis inferiores calculados sobre o subconjunto de análise
q1_pontos  = df_analise['pontos'].quantile(0.25)
q1_minutos = df_analise['minutos_totais'].quantile(0.25)
q1_jogos   = df_analise['jogos_disputados'].quantile(0.25)

print(f'Q1 pontos:          {q1_pontos:.1f}')
print(f'Q1 minutos_totais:  {q1_minutos:.1f}')
print(f'Q1 jogos_disputados:{q1_jogos:.1f}')

# Indicadores binários
df_analise['ind_pontos']  = (df_analise['pontos']          < q1_pontos).astype(int)
df_analise['ind_minutos'] = (df_analise['minutos_totais']  < q1_minutos).astype(int)
df_analise['ind_jogos']   = (df_analise['jogos_disputados'] < q1_jogos).astype(int)

# Variável aleatória discreta X
df_analise['X'] = df_analise['ind_pontos'] + df_analise['ind_minutos'] + df_analise['ind_jogos']

# Evento B: baixo desempenho (X >= 2)
df_analise['B'] = df_analise['X'] >= 2

print(f'\nDistribuição de X:')
print(df_analise['X'].value_counts().sort_index())
print(f'\nCasos com baixo desempenho (B = X >= 2): {df_analise["B"].sum()} ({df_analise["B"].mean():.3f})')

### 6.2 — PMF: P(X = x)

In [ ]:
n = len(df_analise)
pmf = df_analise['X'].value_counts().sort_index() / n
pmf.name = 'P(X = x)'

pmf_df = pmf.reset_index()
pmf_df.columns = ['x', 'P(X = x)']
pmf_df['P(X = x)'] = pmf_df['P(X = x)'].round(4)

print('Tabela PMF — Distribuição de probabilidade de X:')
display(pmf_df)

print(f'\nVerificação: soma das probabilidades = {pmf_df["P(X = x)"].sum():.4f}')

# Gráfico da PMF
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(pmf_df['x'], pmf_df['P(X = x)'], color='steelblue', edgecolor='white', width=0.5)
ax.set_title('PMF de X — Número de indicadores de baixo desempenho')
ax.set_xlabel('x (número de indicadores)')
ax.set_ylabel('P(X = x)')
ax.set_xticks([0, 1, 2, 3])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 — CDF: P(X ≤ x)

In [ ]:
cdf_df = pmf_df.copy()
cdf_df['P(X <= x)'] = cdf_df['P(X = x)'].cumsum().round(4)

print('Tabela CDF — Função de distribuição acumulada de X:')
display(cdf_df)

# Gráfico da CDF
fig, ax = plt.subplots(figsize=(6, 4))
ax.step(cdf_df['x'], cdf_df['P(X <= x)'], where='post', color='steelblue', linewidth=2)
ax.scatter(cdf_df['x'], cdf_df['P(X <= x)'], color='steelblue', zorder=5)
ax.set_title('CDF de X — Função de distribuição acumulada')
ax.set_xlabel('x')
ax.set_ylabel('P(X ≤ x)')
ax.set_xticks([0, 1, 2, 3])
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 6.4 — Média e desvio padrão de X

In [ ]:
# Média: E[X] = sum(x * P(X=x))
media_X = (pmf_df['x'] * pmf_df['P(X = x)']).sum()

# Variância: Var(X) = E[X²] - E[X]²
ex2 = (pmf_df['x']**2 * pmf_df['P(X = x)']).sum()
var_X = ex2 - media_X**2
dp_X = np.sqrt(var_X)

print(f'Média de X    E[X]:   {media_X:.4f}')
print(f'Variância     Var(X): {var_X:.4f}')
print(f'Desvio padrão DP(X):  {dp_X:.4f}')

**Interpretação:** A média de X indica o número esperado de indicadores de baixo desempenho por caso jogador-temporada. Um valor próximo de 0 significa que a maioria dos casos não apresenta múltiplos indicadores simultâneos de baixo desempenho. O desvio padrão mede a variabilidade dessa contagem — quanto maior, mais heterogênea é a distribuição dos indicadores entre os casos.

## Bloco 7 — Parte 3: Probabilidade Condicional

Comparamos P(S | B) e P(S | ¬B) para responder à pergunta de pesquisa.

In [ ]:
def prob(E):
    """Probabilidade de um evento booleano E."""
    return E.sum() / E.size

def prob_cond(A, B):
    """Probabilidade condicional de A dado B."""
    return prob(A[B])

S = df_analise['nao_permanencia']
B = df_analise['B']
nao_B = ~B

p_S       = prob(S)
p_S_dado_B     = prob_cond(S, B)
p_S_dado_naoB  = prob_cond(S, nao_B)
diferenca = p_S_dado_B - p_S_dado_naoB

print(f'P(S)           = {p_S:.4f}  →  probabilidade geral de não permanência')
print(f'P(S | B)       = {p_S_dado_B:.4f}  →  não permanência dado baixo desempenho')
print(f'P(S | ¬B)      = {p_S_dado_naoB:.4f}  →  não permanência dado não baixo desempenho')
print(f'P(S|B) - P(S|¬B) = {diferenca:.4f}')

### 7.1 — Visualização da comparação

In [ ]:
categorias = ['P(S | B)\n(baixo desempenho)', 'P(S | ¬B)\n(não baixo desempenho)', 'P(S)\n(geral)']
valores    = [p_S_dado_B, p_S_dado_naoB, p_S]
cores      = ['tomato', 'steelblue', 'gray']

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(categorias, valores, color=cores, edgecolor='white', width=0.5)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_title('Comparação: P(S | B) vs P(S | ¬B)\nNão permanência observada no mesmo time')
ax.set_ylabel('Probabilidade')
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretação:** A comparação entre P(S | B) e P(S | ¬B) responde diretamente à pergunta de pesquisa. Se P(S | B) > P(S | ¬B), observamos que a não permanência é mais frequente entre casos com baixo desempenho do que entre os demais — o que constitui uma associação observacional positiva entre baixo desempenho e não permanência. A diferença entre essas probabilidades quantifica a magnitude dessa associação. Não é possível concluir, com base nessa análise, que o baixo desempenho *causa* a não permanência — apenas que os dois eventos tendem a ocorrer juntos com mais frequência do que o esperado pela probabilidade marginal.

## Bloco 8 — Análise com Distribuição de Poisson

**Variável de contagem escolhida:** número de indicadores de baixo desempenho (`X`) acumulados por temporada, com a temporada como unidade de exposição.

Estimamos λ como a média de X no subconjunto de análise.

In [ ]:
from scipy.stats import poisson

# Lambda estimado pela média empírica de X
lam = df_analise['X'].mean()
print(f'Lambda estimado (média de X): {lam:.4f}')
print(f'Unidade de exposição: 1 temporada por jogador (caso jogador-temporada)')

x_vals = np.arange(0, 4)
pmf_poisson = poisson.pmf(x_vals, lam)
pmf_empirica = np.array([prob(df_analise['X'] == x) for x in x_vals])

tabela_poisson = pd.DataFrame({
    'x': x_vals,
    'PMF empírica': pmf_empirica.round(4),
    'PMF Poisson(λ)': pmf_poisson.round(4),
})
print('\nComparação PMF empírica vs Poisson:')
display(tabela_poisson)

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(7, 4))
largura = 0.35
ax.bar(x_vals - largura/2, pmf_empirica, largura, label='Empírica', color='steelblue', alpha=0.8)
ax.bar(x_vals + largura/2, pmf_poisson,  largura, label=f'Poisson(λ={lam:.2f})', color='coral', alpha=0.8)
ax.set_title('PMF empírica de X vs distribuição de Poisson')
ax.set_xlabel('x (número de indicadores de baixo desempenho)')
ax.set_ylabel('Probabilidade')
ax.set_xticks(x_vals)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretação:** A distribuição de Poisson modela o número esperado de indicadores de baixo desempenho por caso jogador-temporada, usando como parâmetro λ a média empírica de X. A unidade de exposição é uma temporada por jogador. A comparação entre a PMF empírica e a teórica permite avaliar o ajuste do modelo — desvios indicam que a variável X pode não satisfazer plenamente as hipóteses de Poisson (independência e taxa constante), o que é esperado em dados de desempenho esportivo.

---
## Bloco 9 — Conclusão

A análise de correlação revelou associações positivas entre `pontos`, `minutos_totais`, `jogos_disputados` e `salario_usd` — jogadores com mais participação em quadra tendem a acumular mais pontos e salários mais altos. Essas correlações indicam associação linear, não relação causal.

A variável aleatória discreta X mostrou que a maioria dos casos jogador-temporada apresenta poucos ou nenhum indicador simultâneo de baixo desempenho. Os casos com X = 0 ou X = 1 concentram a maior parte da distribuição, enquanto X = 3 é raro — o que indica que acumular os três indicadores ao mesmo tempo é incomum no dataset.

A comparação entre P(S | B) e P(S | ¬B) sugere que a não permanência observada no mesmo time é mais frequente entre casos classificados como baixo desempenho (B = X ≥ 2) do que entre os demais. Isso constitui uma associação observacional positiva entre baixo desempenho e não permanência.

A análise tem limitações importantes: (1) não é possível afirmar causalidade — a não permanência pode ser influenciada por fatores não observados, como lesões, acordos contratuais ou estratégias de gestão; (2) a variável `nao_permanencia` mede apenas ausência no mesmo time na temporada seguinte, sem distinguir aposentadoria, transferência voluntária ou dispensa; (3) os limiares de baixo desempenho foram definidos com base nos quartis do dataset completo, o que pode não capturar contextos específicos de posição ou papel tático.